# Lab | Neural Networks Fundamentals

**Dataset:** Breast Cancer Wisconsin (scikit-learn built-in)  
**Goal:** Build a small MLP from scratch in NumPy, replicate it in PyTorch, then experiment with activation functions.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
torch.manual_seed(42)
print('torch version:', torch.__version__)


---
## Task 1 — A Single Neuron in NumPy

A single neuron with a sigmoid activation is logistic regression in disguise.


In [ ]:
# 1. Load & prepare data
data = load_breast_cancer()
X, y = data.data, data.target          # (569, 30), (569,)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')


In [ ]:
# 2. Single-neuron forward pass
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Initialise weights from a small normal distribution
rng = np.random.default_rng(42)
w = rng.normal(0, 0.01, size=(30,))
b = float(rng.normal(0, 0.01))

def forward_single(x, w, b):
    """Single neuron forward pass. x: (30,) -> scalar probability."""
    return sigmoid(np.dot(w, x) + b)

# Run on first 5 test rows
probs = np.array([forward_single(X_test[i], w, b) for i in range(5)])
print('Predicted probabilities for first 5 test samples:')
for i, p in enumerate(probs):
    print(f'  sample {i}: {p:.6f}  (true label: {y_test[i]})')


**Interpretation — Task 1:**  
A single neuron with sigmoid activation is exactly **logistic regression**: it computes a weighted linear combination of the 30 input features and squashes the result to a probability in (0, 1). Without any hidden layer there is no non-linear representation capacity beyond what the raw features already provide.


---
## Task 2 — A Two-Layer MLP in NumPy

Architecture: **30 → 8 → 1**, ReLU hidden layer, Sigmoid output, He initialisation.


In [ ]:
class NumpyMLP:
    """Two-layer MLP: input(30) -> hidden(8, ReLU) -> output(1, Sigmoid)."""

    def __init__(self, n_in=30, n_hidden=8, n_out=1, seed=42):
        rng = np.random.default_rng(seed)
        # He initialisation: std = sqrt(2 / fan_in)
        self.W1 = rng.normal(0, np.sqrt(2 / n_in),     size=(n_in,     n_hidden))  # (30, 8)
        self.b1 = np.zeros(n_hidden)                                                # (8,)
        self.W2 = rng.normal(0, np.sqrt(2 / n_hidden), size=(n_hidden, n_out))     # (8, 1)
        self.b2 = np.zeros(n_out)                                                   # (1,)

    @staticmethod
    def _relu(z):
        return np.maximum(0, z)

    @staticmethod
    def _sigmoid(z):
        return 1 / (1 + np.exp(-z))

    def forward(self, X):
        """
        Vectorised forward pass.
        X : (N, 30)
        returns : (N, 1) probabilities
        """
        z1 = X @ self.W1 + self.b1        # (N, 8)
        a1 = self._relu(z1)               # (N, 8)
        z2 = a1 @ self.W2 + self.b2       # (N, 1)
        a2 = self._sigmoid(z2)            # (N, 1)
        return a2


numpy_mlp = NumpyMLP()
numpy_out = numpy_mlp.forward(X_test)    # (114, 1)

print('Output shape:', numpy_out.shape)
print('First 5 predictions:')
for i in range(5):
    print(f'  sample {i}: {numpy_out[i, 0]:.6f}  (true label: {y_test[i]})')


**Interpretation — Task 2:**  
The output shape `(N, 1)` is exactly right for binary classification: one sigmoid-activated probability per sample. The shape is determined solely by the architecture (1 output unit), not by the weights — so it is correct even before any training.

He initialisation sets the weight variance to `2 / fan_in`, which prevents the signal from vanishing or exploding through the ReLU hidden layer at initialisation.


---
## Task 3 — The Same Network in PyTorch

Copy the exact NumPy weights into a `nn.Module` and verify that outputs are numerically identical.


In [ ]:
class TorchMLP(nn.Module):
    """Same architecture as NumpyMLP: 30 -> 8 (ReLU) -> 1 (Sigmoid)."""

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(30, 8)
        self.fc2 = nn.Linear(8, 1)
        self.relu    = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x


torch_mlp = TorchMLP()

# --- Copy weights from NumPy MLP ---
# nn.Linear stores weight as (out_features, in_features) -> we must transpose
with torch.no_grad():
    torch_mlp.fc1.weight.data = torch.from_numpy(numpy_mlp.W1.T).float()  # (8, 30)
    torch_mlp.fc1.bias.data   = torch.from_numpy(numpy_mlp.b1).float()    # (8,)
    torch_mlp.fc2.weight.data = torch.from_numpy(numpy_mlp.W2.T).float()  # (1, 8)
    torch_mlp.fc2.bias.data   = torch.from_numpy(numpy_mlp.b2).float()    # (1,)

# --- Forward pass ---
torch_mlp.eval()
X_test_t = torch.from_numpy(X_test).float()           # (114, 30)
with torch.no_grad():
    torch_out = torch_mlp(X_test_t).numpy()            # (114, 1)

# --- Compare ---
max_diff = np.max(np.abs(numpy_out - torch_out))
print(f'NumPy   output (first 5): {numpy_out[:5, 0]}')
print(f'PyTorch output (first 5): {torch_out[:5, 0]}')
print(f'\nMaximum absolute difference: {max_diff:.2e}')
assert max_diff < 1e-5, 'Outputs do not match!'
print('✓ Outputs match to within numerical precision.')


**Interpretation — Task 3:**  
Both networks produce numerically identical outputs (max absolute difference < 1e-5). The tiny residual is purely float32 rounding — there is no logical difference between the NumPy and PyTorch computations. This confirms that `nn.Linear` is performing the same matrix multiplication as our handwritten NumPy implementation. The key shape gotcha: `nn.Linear` stores `weight` as `(out, in)`, so we must transpose our NumPy weight matrices when copying.


---
## Task 4 — Activation Function Experiment

Compare **Sigmoid**, **Tanh**, and **ReLU** hidden activations by examining the distribution of pre- and post-activations on the test set.


In [ ]:
class FlexMLP(nn.Module):
    """30 -> 8 (configurable activation) -> 1 (Sigmoid). He-initialised."""

    def __init__(self, activation: nn.Module):
        super().__init__()
        self.fc1 = nn.Linear(30, 8)
        self.fc2 = nn.Linear(8, 1)
        self.act     = activation
        self.sigmoid = nn.Sigmoid()
        # He initialisation
        nn.init.kaiming_normal_(self.fc1.weight, nonlinearity='relu')
        nn.init.zeros_(self.fc1.bias)
        nn.init.kaiming_normal_(self.fc2.weight, nonlinearity='relu')
        nn.init.zeros_(self.fc2.bias)

    def forward_with_hidden(self, x):
        """Returns (pre-activation z1, post-activation a1, final output)."""
        z1  = self.fc1(x)
        a1  = self.act(z1)
        out = self.sigmoid(self.fc2(a1))
        return z1, a1, out


torch.manual_seed(42)
variants = {
    'Sigmoid': FlexMLP(nn.Sigmoid()),
    'Tanh'   : FlexMLP(nn.Tanh()),
    'ReLU'   : FlexMLP(nn.ReLU()),
}

# Collect hidden activations
results = {}
X_t = torch.from_numpy(X_test).float()
for name, model in variants.items():
    model.eval()
    with torch.no_grad():
        z1, a1, _ = model.forward_with_hidden(X_t)
    results[name] = {
        'pre':  z1.numpy().flatten(),
        'post': a1.numpy().flatten(),
    }

print('Collected activations for:', list(results.keys()))


In [ ]:
# Plot 1: Pre-activation histograms (z1 — before activation function)
colors = ['#e07b54', '#5b8dd9', '#5caa68']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Hidden-layer PRE-activations (z¹) — before activation function',
             fontsize=13, fontweight='bold')

for ax, (name, color) in zip(axes, zip(results, colors)):
    ax.hist(results[name]['pre'], bins=40, color=color, edgecolor='white', linewidth=0.4)
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('z¹ value')
    ax.set_ylabel('Count')
    ax.axvline(0, color='black', linewidth=1.2, linestyle='--', label='z=0')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('pre_activations.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Plot 2: Post-activation histograms (a1 — after activation function)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Hidden-layer POST-activations (a¹) — after activation function',
             fontsize=13, fontweight='bold')

for ax, (name, color) in zip(axes, zip(results, colors)):
    ax.hist(results[name]['post'], bins=40, color=color, edgecolor='white', linewidth=0.4)
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('a¹ value')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('post_activations.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Saturation / dead-unit statistics
thr = 0.05

sig_post  = results['Sigmoid']['post']
tanh_post = results['Tanh']['post']
relu_post = results['ReLU']['post']

sig_sat  = np.mean((sig_post  < thr) | (sig_post  > 1 - thr))
tanh_sat = np.mean((tanh_post < -1 + thr) | (tanh_post > 1 - thr))
relu_dead = np.mean(relu_post == 0.0)

print(f'Sigmoid — fraction saturated (< {thr} or > {1-thr}): {sig_sat:.2%}')
print(f'Tanh    — fraction saturated (< {-1+thr} or > {1-thr}): {tanh_sat:.2%}')
print(f'ReLU    — fraction inactive (== 0):                     {relu_dead:.2%}')


**Interpretation — Task 4:**

| Activation | Observation |
|------------|-------------|
| **Sigmoid** | A notable fraction of post-activations cluster near 0 or 1 (saturated region). In these zones the derivative ≈ 0, so gradients vanish during back-propagation. |
| **Tanh** | Zero-centred (range −1 to +1), which improves gradient flow slightly vs. Sigmoid, but it still saturates near ±1 for large pre-activations. |
| **ReLU** | Some fraction of units output exactly 0 (dead units), but the *active* units have a constant derivative of 1 — **no saturation in the positive region**. |

**Why ReLU is the better default:**  
For Sigmoid and Tanh, a large fraction of neurons saturate even at initialisation, producing near-zero gradients that slow or stall training (vanishing gradient problem). ReLU avoids this in its active region (z > 0): gradients pass through unchanged, making optimisation faster and enabling deeper networks. Dead units (z ≤ 0) are manageable via He initialisation or leaky variants, and are far less damaging than the near-zero gradients saturated sigmoid/tanh units produce across an entire batch.
